# EVA — Colab Training (T4)

Google Drive → T4 GPU (16 GB) → обучение B=32/ML=256 → автосинк на Drive.

**Перед запуском**: загрузите на `MyDrive/FCF/`:
- `real_data/full_corpus_encoded.npy` (425 MB)
- `checkpoints/full_latest.pt` (22 MB, опционально)
- `checkpoints/full_best.pt` (22 MB, опционально)
- `checkpoints/trajectory_store_full.pkl` (3 MB, опционально)

In [ ]:
# ─── 1. Mount Drive (пропуск, если уже смонтирован) ───
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
print('Drive ready')

In [ ]:
# ─── 2. Clone repo + copy data ───
import shutil
REPO = '/content/FCF'
DRIVE = '/content/drive/MyDrive/FCF'

if not os.path.exists(REPO):
    !git clone https://github.com/BlackCatSpb/FCF.git {REPO}
else:
    !cd {REPO} && git pull
os.chdir(REPO)

for src, dst in [
    (f'{DRIVE}/real_data/full_corpus_encoded.npy', f'{REPO}/real_data/full_corpus_encoded.npy'),
    (f'{DRIVE}/checkpoints/full_latest.pt', f'{REPO}/checkpoints/symbolic/full_latest.pt'),
    (f'{DRIVE}/checkpoints/full_best.pt', f'{REPO}/checkpoints/symbolic/full_best.pt'),
    (f'{DRIVE}/checkpoints/trajectory_store_full.pkl', f'{REPO}/checkpoints/symbolic/trajectory_store_full.pkl'),
]:
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print(f'Copied {os.path.basename(src)}')
print('Data ready')

In [ ]:
# ─── 3. Install deps ───
!pip install numpy scikit-learn --quiet
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.mem_get_info()[1]/1e9:.1f} GB')

In [ ]:
# ─── 4. Colab config (B=32, ML=256) ───
%%writefile /content/FCF/eva/symbolic/colab_config.py
DEVICE = 'cuda'
STEPS = 100000
LR = 5e-3
B = 32
ML = 256
CKPT = '/content/FCF/checkpoints/symbolic'
DRIVE_DIR = '/content/drive/MyDrive/FCF/checkpoints'

In [ ]:
# ─── 5. Run training (автосинк на Drive каждые 5000 шагов) ───
print('>>> Training on Colab T4...')
!python train_full_corpus.py 2>&1

In [ ]:
# ─── 6. Финальный sync + скачать best checkpoint ───
import shutil
DRIVE_SYNC = '/content/drive/MyDrive/FCF/checkpoints'
os.makedirs(DRIVE_SYNC, exist_ok=True)
for fn in ['full_latest.pt', 'full_best.pt', 'trajectory_store_full.pkl']:
    src = f'/content/FCF/checkpoints/symbolic/{fn}'
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_SYNC}/{fn}')
        print(f'Synced {fn}')

from google.colab import files
best = f'{DRIVE_SYNC}/full_best.pt'
if os.path.exists(best):
    files.download(best)
elif os.path.exists('/content/FCF/checkpoints/symbolic/full_best.pt'):
    files.download('/content/FCF/checkpoints/symbolic/full_best.pt')